# Day 068 — Exercise 3: Extract PDF Text

**What you'll build:** `extract_pdf_text(pdf_bytes) -> list[dict]` — page-by-page text extraction from a PDF using pypdf.

**Why it matters:** Text-based PDFs are the most common document type in business workflows. pypdf extracts text instantly — no OCR, no model, no network. The per-page structure lets you detect scanned pages (chars == 0) and route them to an OCR fallback.

In [ ]:
import io
import pypdf
import io, struct

def _minimal_pdf(text: str) -> bytes:
    '''Generate a tiny single-page PDF with the given text (Helvetica 12pt).'''
    lines = text.splitlines()
    tf_lines = ''.join(f'({ln}) Tj T* ' for ln in lines)
    stream = (
        f'BT /F1 12 Tf 50 750 Td {tf_lines}ET'
    ).encode()
    slen = len(stream)

    objs = {
        1: b'<< /Type /Catalog /Pages 2 0 R >>',
        2: b'<< /Type /Pages /Kids [3 0 R] /Count 1 >>',
        3: b'<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Resources << /Font << /F1 << /Type /Font /Subtype /Type1 /BaseFont /Helvetica >> >> >> /Contents 4 0 R >>',
        4: f'<< /Length {slen} >>\nstream\n'.encode() + stream + b'\nendstream',
    }
    buf = io.BytesIO()
    buf.write(b'%PDF-1.4\n')
    offsets = {}
    for num, obj_bytes in objs.items():
        offsets[num] = buf.tell()
        buf.write(f'{num} 0 obj\n'.encode())
        buf.write(obj_bytes)
        buf.write(b'\nendobj\n')

    xref_pos = buf.tell()
    buf.write(b'xref\n')
    buf.write(f'0 {len(objs)+1}\n'.encode())
    buf.write(b'0000000000 65535 f \n')
    for num in range(1, len(objs)+1):
        buf.write(f'{offsets[num]:010d} 00000 n \n'.encode())
    buf.write(
        f'trailer << /Size {len(objs)+1} /Root 1 0 R >>\n'
        f'startxref\n{xref_pos}\n%%EOF\n'.encode()
    )
    return buf.getvalue()

_test_pdf = _minimal_pdf('Hello PDF\nLine two')


## Task

Implement `extract_pdf_text(pdf_bytes: bytes) -> list[dict]`:

1. `reader = pypdf.PdfReader(io.BytesIO(pdf_bytes))`
2. For each page: `text = page.extract_text() or ''`
3. Append `{'page': i, 'text': text, 'chars': len(text)}` (1-based page numbers)
4. Return the list

A minimal in-memory PDF is provided — no file system needed.

## Your Implementation

In [ ]:
def extract_pdf_text(pdf_bytes: bytes) -> list:
    """Extract text from each page of a PDF.

    Args:
        pdf_bytes: raw PDF bytes
    Returns:
        List of dicts: [{page: int (1-based), text: str, chars: int}]
    """
    raise NotImplementedError


In [ ]:
def extract_pdf_text(pdf_bytes: bytes) -> list:
    reader = pypdf.PdfReader(io.BytesIO(pdf_bytes))
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        pages.append({'page': i, 'text': text, 'chars': len(text)})
    return pages


## Automated checks

In [ ]:
score, total = 0, 5
try:
    result = extract_pdf_text(_test_pdf)
    assert isinstance(result, list), f"Expected list, got {type(result)}"
    score += 1; print("\u2705 returns a list")

    assert len(result) == 1, f"Expected 1 page, got {len(result)}"
    score += 1; print("\u2705 returns one entry per page")

    page = result[0]
    assert 'page' in page and 'text' in page and 'chars' in page, (
        f"Missing keys: {set(page.keys())}")
    score += 1; print("\u2705 each entry has 'page', 'text', 'chars' keys")

    assert page['page'] == 1, f"Page number should be 1-based, got {page['page']}"
    score += 1; print("\u2705 page numbering is 1-based")

    assert isinstance(page['text'], str), "text should be a string"
    assert page['chars'] == len(page['text']), (
        f"chars {page['chars']} != len(text) {len(page['text'])}")
    score += 1; print("\u2705 chars equals len(text)")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def extract_pdf_text(pdf_bytes: bytes) -> list:
    reader = pypdf.PdfReader(io.BytesIO(pdf_bytes))
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        pages.append({'page': i, 'text': text, 'chars': len(text)})
    return pages
```

**Why `or ''`?** `page.extract_text()` returns `None` for pages that contain no text objects (image-only / scanned pages). Without the guard, `len(None)` raises a `TypeError`. The `or ''` converts `None` to an empty string, and `chars == 0` signals a scanned page that needs OCR fallback.

</details>